In [1]:
pip install unidecode

In [2]:
pip install contractions

In [3]:
def text_polishing(corpus):
  elaborated_texts=[]
  for text in corpus:
    expanded_text = contractions.fix(text) # Expansion of contractions didn't-> did not
    text = expanded_text.split()

    cleaned_words = []
    for word in text:
      clean_word = word.lower()   #Lowercasing words
      clean_word = "".join([i for i in clean_word if i not in string.punctuation]) #Remove punctuation
      clean_word = unidecode(clean_word) #Normalize accented characters

      if clean_word and clean_word.isalpha() and clean_word not in stop_words: #Filtered removing stopwords
          cleaned_words.append(clean_word)
    elaborated_texts.append(" ".join(cleaned_words))
  return elaborated_texts

def top_tokens_extraction(vectorizer, final_clf):
  feature_names = vectorizer.get_feature_names_out()

  model_weights = final_clf.coef_[0]

  word_features = list(zip(model_weights, feature_names))
  ordered_words = sorted(word_features, key=lambda x: abs(x[0]), reverse=True)[:30]
  return ordered_words

def optimal_model(X_train_vec, y_train, X_val_vec, y_val):
  best_acc = 0
  best_c=0
  best_model = None

  for c_val in [0.01, 0.1, 0.15, 0.5, 1, 10]:
      clf = LogisticRegression(C=c_val, random_state=42, max_iter=1000)
      clf.fit(X_train_vec, y_train)

      score = accuracy_score(y_val, clf.predict(X_val_vec))
      if score > best_acc:
          best_acc = score
          best_c = c_val
          best_model = clf
  return best_c, best_model

In [7]:
import random
import spacy
from nltk.corpus import movie_reviews
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from unidecode import unidecode
import contractions
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import nltk
nltk.download('movie_reviews') # loads the dataset
nltk.download('punkt')
!python -m spacy download "en_core_web_sm"
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


nlp_en = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

documents = [(movie_reviews.raw(fileid), category)
              for category in movie_reviews.categories()
              for fileid in movie_reviews.fileids(category)]

print("\nNumber of docs loaded:", len(documents))

corpus_raw = [ x[0] for x in documents ]  # corpus_raw is a list of strings (reviews) to be converted into vectors
y_corpus = [ x[1] for x in documents ]    # y_corpus are the sentiment labels of the reviews (nothing to be done here)

# printing an example document and its class
#print(corpus_raw[0])  # notice the noise in the review (especially the last row)
#print(y_corpus[0])    # the class is "neg", but scikit learn is able to handle non numerical labels automatically

random.seed(42)

#Data splitting.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    corpus_raw, y_corpus, test_size=0.3, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42
)

#BoW vectorization
vectorizer = CountVectorizer(stop_words='english', max_features=5000)

#Pre-Processing of training and validation sets.
train_polished = text_polishing(X_train)
val_polished = text_polishing(X_val)

#Vectorization (Dictionary creation) of training and validation sets.
X_train_vec = vectorizer.fit_transform(train_polished)
X_val_vec = vectorizer.transform(val_polished)


#Hyper-parameter optimization.
best_c, best_model = optimal_model(X_train_vec, y_train, X_val_vec, y_val)
print(f"Optimal C found: {best_c}")


#Re-union of training and validation sets. Same with labels
#Final processing, vectorization and evaluation on test set.

full_train_polished = train_polished + val_polished
y_train_full_list = list(y_train) + list(y_val)

test_polished = text_polishing(X_test)

vectorizer_final = CountVectorizer(stop_words='english', max_features=5000)
X_train_full_vec = vectorizer_final.fit_transform(full_train_polished)
X_test_vec_final = vectorizer_final.transform(test_polished)

final_clf = LogisticRegression(C=best_c, random_state=42, max_iter=1000)
final_clf.fit(X_train_full_vec, y_train_full_list)

y_pred = final_clf.predict(X_test_vec_final)

print(f"Final Accuracy on test set: {accuracy_score(y_test, y_pred):.4f}")
print("\n--- Detailed Performance Report ---")
print(classification_report(y_test, y_pred))

print("Top 30 tokens by absolute parameter value:")
top_tokens = top_tokens_extraction(vectorizer_final, final_clf)
for weight, word in top_tokens:
    print(f"{word:<15} | weight: {weight:.4f}")

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 40.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Number of docs loaded: 2000
Optimal C found: 0.15
Final Accuracy on test set: 0.8133

--- Detailed Performance Report ---
              precision    recall  f1-score   support

         neg       0.81      0.82      0.82       302
         pos       0.82      0.81      0.81       298

    accuracy                           0.81       600
   macro avg       0.81      0.81      0.81       600
weighted avg       0.81      0.81      0.81       600

Top 30 tokens by absolute parameter value:
bad             | weight: -0.6378
plot            | weight: -0.4529
worst           | weight: -0.4325
supposed        | weight: -0.4095
great           | weight: 0.4042
fun             | weight: 0.3987
perfectly       | weight: 0.3368
poor            | weight: -0.3362
unfortunately   | weight: -0.3349
entertaining    | weight: 0.3330
stupid          | weight: -0.3315
boring          | weight: -0.3301
looks           | weight: -0.3241
performances    | weight: 0.3193
seen            | weight: 0.3163
ove